# Analysis workflow for superalloy oxidation

Public steps run from released CSV files. Controlled steps use author-supplied modeling data and a separate working directory. The five archived training cells retain their original training logs with machine-specific paths removed. Set EVIDENCE_RUN_TRAINING=1 to retrain.


### Configure the analysis environment

Locates the package and optionally configures an external workspace for controlled data and detailed logs.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

root = next(p.resolve() for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "evidence_aware_ML")
            if (p / "code/verify_published_evidence.py").is_file())
code = root / "code"
work_value = os.environ.get("EVIDENCE_WORK_DIR")
work = Path(work_value).expanduser().resolve() if work_value else None
modeling_value = os.environ.get("EVIDENCE_MODELING_CSV")
modeling_csv = Path(modeling_value).expanduser().resolve() if modeling_value else None
run_training = os.environ.get("EVIDENCE_RUN_TRAINING") == "1"
run_fitting_quality = os.environ.get("EVIDENCE_RUN_FITTING_QUALITY") == "1"
if work is not None:
    if work.is_relative_to(root):
        raise ValueError("EVIDENCE_WORK_DIR must be outside the release package.")
    work.mkdir(parents=True, exist_ok=True)
public_output = Path(tempfile.mkdtemp(prefix="evidence_public_output_"))

def run_script(relative_path, *args, controlled=False, enabled=True):
    script = code / relative_path
    if not enabled or (controlled and work is None):
        print(f"Skipped {script.name}: required inputs or execution flag are unavailable.")
        return
    log_folder = (work if controlled else public_output) / "logs"
    log_folder.mkdir(exist_ok=True)
    log_path = log_folder / (script.stem + ".log")
    env = dict(os.environ, PYTHONPATH=str(code), PYTHONDONTWRITEBYTECODE="1",
               PYTHONIOENCODING="utf-8", MPLBACKEND="Agg")
    with log_path.open("a", encoding="utf-8") as log:
        result = subprocess.run([sys.executable, str(script), *map(str, args)],
                                cwd=work if controlled else root, env=env,
                                stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f"{script.name} exited with code {result.returncode}. "
                           "See the detailed log in the working directory.")
    print(f"{script.name} completed. Outputs are in the working directory.")


### Verify published evidence

Runs verify_published_evidence.py. Checks literature and fold membership, nested thresholds, temporal states, and candidate pairing.

In [2]:
run_script('verify_published_evidence.py', "--output-dir", public_output / "tables")

verify_published_evidence.py completed. Outputs are in the working directory.


### Plot the prospective screening panels

Runs plot_prospective_screening.py. Rebuilds Fig. 6(a–d) from virtual candidate results.

In [3]:
run_script('figures/plot_prospective_screening.py', "--output-dir", public_output / "figures")

plot_prospective_screening.py completed. Outputs are in the working directory.


### Restore ordered validation folds

Runs prepare_validation_splits.py. Prepares the two CV schemes and the deployment reference from the processed modeling input.

In [4]:
run_script('preprocessing/prepare_validation_splits.py', "--modeling-data", modeling_csv, "--work-dir", work, controlled=True, enabled=modeling_csv is not None)

prepare_validation_splits.py completed. Outputs are in the working directory.


### Train nested source-aware XGBoost

Runs train_source_aware_xgboost.py. Performs inner optimization and outer evaluation with literature sources as groups.

In [6]:
run_script('nested_cv/train_source_aware_xgboost.py', controlled=True, enabled=run_training)

NESTED 5-FOLD SOURCE-AWARE XGBOOST / OPTUNA
Outer folds=5, inner folds=4, trials/outer=100
Features=32, source column=source_key
Original models and MED/PED pipeline will not be modified.

========================= OUTER FOLD 1/5 =========================
Outer train: 490 rows / 130 sources
Outer test : 123 rows / 33 sources
Source overlap between outer train/test: 0
Optuna completed: 0; running: 100 new trials
  Outer 1: 1/100 trials; best inner mean R2=0.26162
  Outer 1: 10/100 trials; best inner mean R2=0.65795
  Outer 1: 20/100 trials; best inner mean R2=0.65795
  Outer 1: 30/100 trials; best inner mean R2=0.65964
  Outer 1: 40/100 trials; best inner mean R2=0.65964
  Outer 1: 50/100 trials; best inner mean R2=0.66825
  Outer 1: 60/100 trials; best inner mean R2=0.66825
  Outer 1: 70/100 trials; best inner mean R2=0.66825
  Outer 1: 80/100 trials; best inner mean R2=0.66825
  Outer 1: 90/100 trials; best inner mean R2=0.66825
  Outer 1: 100/100 trials; best inner mean R2=0.66914
Be

### Train nested random XGBoost

Runs train_random_xgboost.py. Performs the matching comparison with record-level random splits.

In [1]:
run_script('nested_cv/train_random_xgboost.py', controlled=True, enabled=run_training)

NESTED 5-FOLD RANDOM XGBOOST / OPTUNA (MATCHED TO SOURCE-AWARE NESTED)
Outer=5 RANDOM KFold saved splits; inner=4 shuffled RANDOM KFold; trials/outer=100
Features=32, objective=mean inner R2, n_jobs=8
source_key is used for auditing ONLY; no source grouping in random CV
No original models or evidence files will be modified

========================= OUTER RANDOM FOLD 1/5 =========================
Outer train=490 / 157 sources; test=123 / 83 sources; overlapping sources=77
Optuna completed=0; new trials=100
  Fold 1: 1/100; best inner mean R2=0.32707
  Fold 1: 10/100; best inner mean R2=0.79001
  Fold 1: 20/100; best inner mean R2=0.80131
  Fold 1: 30/100; best inner mean R2=0.80439
  Fold 1: 40/100; best inner mean R2=0.80439
  Fold 1: 50/100; best inner mean R2=0.81254
  Fold 1: 60/100; best inner mean R2=0.81987
  Fold 1: 70/100; best inner mean R2=0.82138
  Fold 1: 80/100; best inner mean R2=0.83001
  Fold 1: 90/100; best inner mean R2=0.83330
  Fold 1: 100/100; best inner mean R2=0

### Benchmark the source-aware regression models

Runs benchmark_regression_models.py. Computes the source-aware metrics for Table S3.

In [21]:
run_script('nested_cv/benchmark_regression_models.py', "--split", "source", controlled=True, enabled=run_training)

NESTED 6-MODEL BENCHMARK (XGBOOST PRECOMPUTED)
split=source; models=['LR', 'KNN', 'SVR', 'RF', 'MLP']; inner=4, trials/model/outer=100
All Optuna selection is inside each outer TRAIN set. No outer test tuning.

SOURCE nested benchmark: 613 OOF records, models=['LR', 'KNN', 'SVR', 'RF', 'MLP']
 source LR fold1: REUSE saved outer prediction
 source LR fold2: REUSE saved outer prediction
 source LR fold3: REUSE saved outer prediction
 source LR fold4: REUSE saved outer prediction
 source LR fold5: REUSE saved outer prediction
  FINAL source/LR: pooled R2=0.56875 MAE=0.71239
 source KNN fold1: Optuna complete=100, new=0
 source KNN fold1: REUSE saved outer prediction
 source KNN fold2: Optuna complete=100, new=0
 source KNN fold2: REUSE saved outer prediction
 source KNN fold3: Optuna complete=100, new=0
 source KNN fold3: REUSE saved outer prediction
 source KNN fold4: Optuna complete=100, new=0
 source KNN fold4: REUSE saved outer prediction
 source KNN fold5: Optuna complete=100, new=0


### Benchmark the random regression models

Runs benchmark_regression_models.py. Computes the random-fold metrics for Table S3.

In [22]:
run_script('nested_cv/benchmark_regression_models.py', "--split", "random", controlled=True, enabled=run_training)

NESTED 6-MODEL BENCHMARK (XGBOOST PRECOMPUTED)
split=random; models=['LR', 'KNN', 'SVR', 'RF', 'MLP']; inner=4, trials/model/outer=100
All Optuna selection is inside each outer TRAIN set. No outer test tuning.

RANDOM nested benchmark: 613 OOF records, models=['LR', 'KNN', 'SVR', 'RF', 'MLP']
 random LR fold1: outer R2=0.64675 MAE=0.66346 RMSE=0.89135
 random LR fold2: outer R2=0.77242 MAE=0.54831 RMSE=0.67642
 random LR fold3: outer R2=0.72050 MAE=0.55294 RMSE=0.73931
 random LR fold4: outer R2=0.55509 MAE=0.65796 RMSE=0.87233
 random LR fold5: outer R2=0.71635 MAE=0.56992 RMSE=0.70597
  FINAL random/LR: pooled R2=0.68884 MAE=0.59847
 random KNN fold1: Optuna complete=0, new=100
 random KNN fold1: 1/100, inner best R2=0.50597
 random KNN fold1: 10/100, inner best R2=0.58827
 random KNN fold1: 20/100, inner best R2=0.65443
 random KNN fold1: 30/100, inner best R2=0.65443
 random KNN fold1: 40/100, inner best R2=0.65443
 random KNN fold1: 50/100, inner best R2=0.65443
 random KNN fold1:

### Calculate outer-fold evidence

Runs compute_oof_evidence.py. Computes MED, PED and fold-specific thresholds with the fitted nested models.

In [9]:
run_script('nested_cv/compute_oof_evidence.py', "--main-weights-json", root / "configs/physical_priors.json", "--skip-diagnostic-check", "--no-excel", controlled=True)

compute_oof_evidence.py completed. Outputs are in the working directory.


### Plot evidence-state error distributions

Runs plot_evidence_error_distributions.py. Rebuilds Fig. 3(d) from the controlled OOF absolute errors.

In [10]:
run_script('figures/plot_evidence_error_distributions.py', controlled=True)

plot_evidence_error_distributions.py completed. Outputs are in the working directory.


### Evaluate evidence-weight sensitivity

Runs evaluate_evidence_weights.py. Varies MED mixing and the PED protocol contribution for Table S5.

In [11]:
run_script('sensitivity_and_ablation/evaluate_evidence_weights.py', "--main-weights-json", root / "configs/physical_priors.json", "--no-excel", controlled=True)

evaluate_evidence_weights.py completed. Outputs are in the working directory.


### Evaluate neighborhood size and applicability ablation

Runs evaluate_neighborhood_ablation.py. Compares k and the distance definitions in Tables S5 and S7.

In [12]:
run_script('sensitivity_and_ablation/evaluate_neighborhood_ablation.py', "--main-weights-json", root / "configs/physical_priors.json", "--no-plot", controlled=True)

evaluate_neighborhood_ablation.py completed. Outputs are in the working directory.


### Fit the cross-fitted MED reference

Runs fit_cross_fitted_med_reference.py. Fits source-disjoint auxiliary models using the saved outer hyperparameters.

In [13]:
run_script('sensitivity_and_ablation/fit_cross_fitted_med_reference.py', controlled=True)

fit_cross_fitted_med_reference.py completed. Outputs are in the working directory.


### Evaluate MED calibration

Runs evaluate_med_calibration.py. Maps held-source ranks and estimates source-cluster intervals for Table S6.

In [14]:
run_script('sensitivity_and_ablation/evaluate_med_calibration.py', "--no-excel", controlled=True)

evaluate_med_calibration.py completed. Outputs are in the working directory.


### Summarize Q4 coverage

Runs summarize_q4_coverage.py. Computes alloy-family and Cr–Al bin statistics for Table S8.

In [15]:
run_script('sensitivity_and_ablation/summarize_q4_coverage.py', controlled=True)

summarize_q4_coverage.py completed. Outputs are in the working directory.


### Summarize kinetic fitting metadata

Runs summarize_fitting_metadata.py. Checks the fitting metadata used in the quality subset analysis.

In [16]:
run_script('sensitivity_and_ablation/summarize_fitting_metadata.py', controlled=True, enabled=run_fitting_quality)

summarize_fitting_metadata.py completed. Outputs are in the working directory.


### Evaluate kinetic fitting quality

Runs evaluate_fitting_quality.py. Compares fixed predictions across subsets defined by the designated fitting metadata.

In [17]:
run_script('sensitivity_and_ablation/evaluate_fitting_quality.py', controlled=True, enabled=run_fitting_quality)

evaluate_fitting_quality.py completed. Outputs are in the working directory.


### Fit paired historical fold models

Runs fit_historical_fold_models.py. Fits the historical models on pre-2023 outer training records with the paired contemporary settings.

In [12]:
run_script('temporal_evolution/fit_historical_fold_models.py', "--no-excel", controlled=True, enabled=run_training)

NESTED SOURCE-AWARE HISTORICAL TRAINING: FOLD-SPECIFIC FROZEN THETA
Cutoff: Year <= 2022; fixed TEST rows kept at all publication years
Fixed current OOF strict verification: True
No Optuna. No MED/PED. No changes to original model/evidence files.

====================== HISTORICAL OUTER FOLD 1/5 ======================
  Saved/current nested OOF & formal evidence: STRICT PASS
  Historical train=334 (82 sources); full train=490, fixed test=123
  HISTORICAL | R2=0.59684 | MAE=0.75396 | RMSE=0.98201
  CURRENT    | R2=0.65167 | MAE=0.70951 | RMSE=0.91280
  Year missing in full train (excluded from historical): 0

====================== HISTORICAL OUTER FOLD 2/5 ======================
  Saved/current nested OOF & formal evidence: STRICT PASS
  Historical train=345 (86 sources); full train=490, fixed test=123
  HISTORICAL | R2=0.66667 | MAE=0.62213 | RMSE=0.81191
  CURRENT    | R2=0.63913 | MAE=0.64716 | RMSE=0.84479
  Year missing in full train (excluded from historical): 0

===============

### Calculate historical and current evidence

Runs compute_temporal_evidence.py. Pairs each held-out record's evidence states and rebuilds the transition matrix.

In [19]:
run_script('temporal_evolution/compute_temporal_evidence.py', "--main-weights-json", root / "configs/physical_priors.json", "--no-excel", controlled=True)

compute_temporal_evidence.py completed. Outputs are in the working directory.


### Summarize temporal transitions

Runs summarize_temporal_transitions.py. Computes transition statistics and traces the literature neighbors for s279 and s125.

In [20]:
run_script('temporal_evolution/summarize_temporal_transitions.py', "--no-excel", "--trace-sample-ids", "s279", "s125", controlled=True)

summarize_temporal_transitions.py completed. Outputs are in the working directory.


### Plot Cr–Al and temporal evidence

Runs plot_spatial_and_temporal_evidence.py. Rebuilds Figs. 4, 5(a–c), and S3 from controlled outputs.

In [21]:
run_script('temporal_evolution/plot_spatial_and_temporal_evidence.py', controlled=True)

plot_spatial_and_temporal_evidence.py completed. Outputs are in the working directory.


### Plot temporal cases

Runs plot_temporal_cases.py. Rebuilds Fig. 5(d) and the physical neighbors in Table S10.

In [22]:
run_script('temporal_evolution/plot_temporal_cases.py', "--no-excel", controlled=True)

plot_temporal_cases.py completed. Outputs are in the working directory.


### Train the independent historical model

Runs validate_chronological_holdout.py. Tunes and fits on literature available at the chronological cutoff.

In [23]:
run_script('chronological_validation/validate_chronological_holdout.py', "train", "--no-overwrite", controlled=True, enabled=run_training)

Skipped validate_chronological_holdout.py: required inputs or execution flag are unavailable.


### Evaluate the future literature holdout

Runs validate_chronological_holdout.py. Scores future records and computes matched-budget screening results.

In [24]:
run_script('chronological_validation/validate_chronological_holdout.py', "evaluate", "--no-overwrite", controlled=True, enabled=work is not None and not (work / "chronological_runs/cutoff_2022_p50/evaluation").exists())

validate_chronological_holdout.py completed. Outputs are in the working directory.


### Analyze chronological prediction errors

Runs analyze_chronological_error.py. Computes stratified and leave-one-source-out error comparisons.

In [25]:
run_script('chronological_validation/analyze_chronological_error.py', controlled=True)

analyze_chronological_error.py completed. Outputs are in the working directory.


### Evaluate screening budget sensitivity

Runs evaluate_screening_budget_sensitivity.py. Varies the candidate-selection budget and prediction tolerance.

In [26]:
run_script('chronological_validation/evaluate_screening_budget_sensitivity.py', controlled=True)

evaluate_screening_budget_sensitivity.py completed. Outputs are in the working directory.


### Fit the full-data deployment model

Runs fit_deployment_model.py. Refits one XGBoost model using the original five-fold optimum without a new Optuna search.

In [27]:
run_script('prospective_screening/fit_deployment_model.py', "--reference", "full_reference.csv", "--model", "deployment.joblib", controlled=True, enabled=work is not None and not (work / "deployment.joblib").exists())

fit_deployment_model.py completed. Outputs are in the working directory.


### Calculate virtual candidate evidence

Runs compute_candidate_evidence.py. Recomputes deployment-model PED and MED and exports Table 3, Table S11, and Fig. 6 inputs.

In [28]:
run_script('prospective_screening/compute_candidate_evidence.py', "--reference", "full_reference.csv", "--model", "deployment.joblib", "--output-dir", "prospective_outputs", controlled=True)

compute_candidate_evidence.py completed. Outputs are in the working directory.
